# Band alignment and wavefunctions: InAs/GaAs pyramidal dot

Visualizations of the corrected strain model (`strain_fourier`), the Bir-Pikus coupling
(`kp_pryor`), and the piezoelectric potential (`piezoelectric`), for Pryor's b = 14 nm
pyramid.

Everything is computed on a **verified-clean grid**: `qdsolver_core.centered_axis` with the
mirror-symmetry and volume checks asserted before any solve. That matters here more than
usual, because the p-state splitting these figures show is a symmetry-breaking effect, and an
asymmetric grid fakes one. At h = 0.7 with `np.arange` the discretized pyramid is lopsided by
180 voxels and 14% undersized; h = 0.5 is exact to 0.1%.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, TwoSlopeNorm

import qdsolver_core as qd
import strain_fourier as sf
import piezoelectric as pz
import kp_confined as kpc
import kp_pryor as kp
import pryor1998 as pr
import eigensolvers as eig

%matplotlib inline

# --- palette (validated: worst all-pairs CVD dE 9.2, normal-vision 24.0) -------------
INK, INK2, MUTED = '#0b0b0b', '#52514e', '#8a8880'
S1, S2, S3 = '#2a78d6', '#eb6834', '#1baf7a'      # categorical slots 1-3, fixed order
SURFACE, GRIDC = '#fcfcfb', '#e6e5e1'

# sequential: ONE hue, light -> dark (documented blue ramp) -- for |psi|^2, a magnitude
SEQ = LinearSegmentedColormap.from_list('seq_blue', [
    '#fcfcfb', '#cde2fb', '#9ec5f4', '#5598e7', '#2a78d6', '#1c5cab', '#0d366b'])
# diverging: two hues + NEUTRAL GRAY midpoint -- for signed fields (strain, potential)
DIV = LinearSegmentedColormap.from_list('div_br', [
    '#0d366b', '#2a78d6', '#9ec5f4', '#f0efec', '#f0a49c', '#e34948', '#8f2322'])

plt.rcParams.update({
    'figure.facecolor': SURFACE, 'axes.facecolor': SURFACE,
    'savefig.facecolor': SURFACE,
    'axes.edgecolor': GRIDC, 'axes.labelcolor': INK2, 'axes.titlecolor': INK,
    'xtick.color': MUTED, 'ytick.color': MUTED,
    'text.color': INK, 'axes.grid': True, 'grid.color': GRIDC,
    'grid.linewidth': 0.6, 'axes.spines.top': False, 'axes.spines.right': False,
    'font.size': 9, 'axes.titlesize': 10, 'lines.linewidth': 2.0, 'figure.dpi': 120,
})
print('ready')

In [ ]:
dot, matrix = pr.PRYOR_TABLE_I['InAs'], pr.PRYOR_TABLE_I['GaAs']
eps_star = qd.eigenstrain(dot['a0'], matrix['a0'])
nu = qd.voigt_poisson_ratio(matrix['C11'], matrix['C12'], matrix['C44'])
CIJ = (matrix['C11'], matrix['C12'], matrix['C44'])
EPS_R, BASE, GAAS_CB = matrix['eps_R'], 14.0, matrix['Eg']
EXACT_VOL = BASE**2 * (BASE / 2) / 3

def clean_grid(h, pad, base=BASE):
    nx = int(round(2 * (base / 2 + pad) / h)) // 2 * 2 + 1
    kz0 = int(round(pad / h))
    nz = kz0 + int(round((base / 2 + pad) / h)) + 1
    cx = qd.centered_axis(nx, h)
    cz = (np.arange(nz) - kz0) * h
    X, Y, Z = np.meshgrid(cx, cx, cz, indexing='ij')
    mask = qd.pyramid_mask(X, Y, Z, base)
    # the guards: an asymmetric grid would fake the very splitting we are plotting
    assert qd.mirror_asymmetry(mask, 0) == 0, 'grid asymmetric in x'
    assert qd.mirror_asymmetry(mask, 1) == 0, 'grid asymmetric in y'
    return cx, cz, X, Y, Z, mask

H, PAD = 0.5, 10.0
cx, cz, X, Y, Z, pyr = clean_grid(H, PAD)
strain = sf.solve_strain(pyr, eps_star, *CIJ, H)
phi = pz.potential(strain, np.where(pyr, dot['e14'], matrix['e14']), EPS_R, H)

ix0, iy0 = len(cx) // 2, len(cx) // 2      # x = y = 0
iz_base = int(np.argmin(np.abs(cz)))        # z = 0, the pyramid base

print(f"grid {X.shape} = {X.size:,} points, {pyr.sum():,} in the dot")
print(f"volume error {qd.mask_volume_error(pyr, H, EXACT_VOL):+.4f}, "
      f"clamping residual {sf.padding_report(strain, pyr):.4f}")
print(f"trace in dot {strain.trace[pyr].mean():+.5f}, "
      f"piezo phi {phi.min()*1e3:+.1f} .. {phi.max()*1e3:+.1f} meV")

## 1. Band alignment

The conduction and valence band edges on Pryor's energy scale (zero = unstrained GaAs valence
band). Three curves per panel: the **unstrained** offsets, the **old hydrostatic-only** model,
and the **corrected** elastic strain.

The old model overstates the trace by the factor $(1+\nu)/(2(1-2\nu)) = 1.165$ AND spreads it
uniformly over the dot with exactly zero in the barrier. The corrected trace varies through the
dot and leaks into the matrix, which is why the corrected well *tapers* from base to apex the
way Pryor's Fig. 4 does.

In [ ]:
tr_new = strain.trace
tr_old, _ = qd.trace_strain_from_mask(pyr, eps_star, nu)
tr_zero = np.zeros_like(tr_new)

fields = {}
for name, tr in (('unstrained', tr_zero), ('old model', tr_old), ('corrected', tr_new)):
    Ve, Vh, me, mh = pr.band_edge_fields(pyr, tr)
    fields[name] = (Ve, Vh)

styles = {'unstrained': (MUTED, '--'), 'old model': (S2, '-'), 'corrected': (S1, '-')}

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))

# (a) vertical cut through the apex
ax = axes[0]
for name, (Ve, Vh) in fields.items():
    c, ls = styles[name]
    ax.plot(cz, Ve[ix0, iy0, :], color=c, ls=ls)
    ax.plot(cz, Vh[ix0, iy0, :], color=c, ls=ls)
    ax.annotate(name, (cz[iz_base + 2], Ve[ix0, iy0, iz_base + 2]), color=c,
                fontsize=8, fontweight='bold', xytext=(3, 4), textcoords='offset points')
ax.axvspan(0, BASE / 2, color=GRIDC, alpha=0.5, zorder=0, lw=0)
ax.annotate('dot', (BASE / 4, 1.62), color=MUTED, fontsize=8, ha='center')
ax.set_xlabel('z (nm), through the apex'); ax.set_ylabel('energy (eV)')
ax.set_title('(a) vertical cut, x = y = 0')
ax.set_xlim(-6, 12); ax.set_ylim(-0.15, 1.75)

# (b) lateral cut at the base
ax = axes[1]
for name, (Ve, Vh) in fields.items():
    c, ls = styles[name]
    ax.plot(cx, Ve[:, iy0, iz_base], color=c, ls=ls)
    ax.plot(cx, Vh[:, iy0, iz_base], color=c, ls=ls)
ax.axvspan(-BASE / 2, BASE / 2, color=GRIDC, alpha=0.5, zorder=0, lw=0)
ax.set_xlabel('x (nm), at the dot base'); ax.set_ylabel('energy (eV)')
ax.set_title('(b) lateral cut, y = 0, z = 0')
ax.set_xlim(-14, 14); ax.set_ylim(-0.15, 1.75)
for ax in axes:
    ax.annotate('$E_c$', (ax.get_xlim()[0] + 0.4, 1.53), color=INK2, fontsize=9)
    ax.annotate('$E_v$', (ax.get_xlim()[0] + 0.4, 0.03), color=INK2, fontsize=9)
fig.suptitle('Band edges on Pryor\'s scale (0 = unstrained GaAs valence band)',
             fontsize=11, y=1.02)
fig.tight_layout(); plt.show()

Ve_c = fields['corrected'][0]; Ve_o = fields['old model'][0]
print(f"CB well depth, corrected: {(GAAS_CB-Ve_c[pyr].mean())*1e3:.1f} meV "
      f"(base {(GAAS_CB-Ve_c[:,:,iz_base][pyr[:,:,iz_base]].mean())*1e3:.1f}, "
      f"old model {(GAAS_CB-Ve_o[pyr].mean())*1e3:.1f})")

### The well profile in the plane

The corrected conduction-band edge as a 2D map. The taper along $z$ and the strain leaking into
the barrier are both visible; the old model would be a flat plateau inside the dot outline and
identically zero shift outside it.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.0))
for ax, (Ve, title) in zip(axes, [(fields['corrected'][0], '(a) corrected'),
                                  (fields['old model'][0], '(b) old hydrostatic-only')]):
    sl = Ve[:, iy0, :].T
    im = ax.pcolormesh(cx, cz, sl, cmap=SEQ, shading='nearest',
                       vmin=0.95, vmax=1.55, rasterized=True)
    ax.contour(cx, cz, pyr[:, iy0, :].T.astype(float), levels=[0.5],
               colors=[INK], linewidths=1.2)
    ax.set_xlabel('x (nm)'); ax.set_ylabel('z (nm)')
    ax.set_title(title + ' — conduction band edge')
    ax.set_xlim(-14, 14); ax.set_ylim(-6, 12); ax.set_aspect('equal'); ax.grid(False)
    fig.colorbar(im, ax=ax, label='$E_c$ (eV)', fraction=0.046)
fig.tight_layout(); plt.show()

## 2. The strain tensor

What the hydrostatic-only model could not represent. The trace is signed, so it uses a
**diverging** ramp with a neutral midpoint; the shear components likewise. The shear is not a
correction — its rms is ~13% of the trace, and it is what the Bir-Pikus $b$ and $d$
deformation potentials couple to.

In [ ]:
comps = [(strain.trace, r'$\mathrm{Tr}\,\varepsilon$', 'xz'),
         (strain.biaxial, r'$\varepsilon_{xx}+\varepsilon_{yy}-2\varepsilon_{zz}$', 'xz'),
         (strain.ezx, r'$\varepsilon_{zx}$', 'xz'),
         (strain.exy, r'$\varepsilon_{xy}$', 'xy')]
fig, axes = plt.subplots(1, 4, figsize=(15, 3.4))
for ax, (fld, label, plane) in zip(axes, comps):
    if plane == 'xz':
        sl, hor, ver = fld[:, iy0, :].T, cx, cz
        outline = pyr[:, iy0, :].T.astype(float)
        ax.set_xlabel('x (nm)'); ax.set_ylabel('z (nm)')
        ax.set_xlim(-14, 14); ax.set_ylim(-6, 12)
    else:
        zi = iz_base + int(round(1.5 / H))
        sl, hor, ver = fld[:, :, zi].T, cx, cx
        outline = pyr[:, :, zi].T.astype(float)
        ax.set_xlabel('x (nm)'); ax.set_ylabel('y (nm)')
        ax.set_xlim(-12, 12); ax.set_ylim(-12, 12)
    v = np.abs(sl).max()
    im = ax.pcolormesh(hor, ver, sl, cmap=DIV, shading='nearest',
                       norm=TwoSlopeNorm(0.0, -v, v), rasterized=True)
    ax.contour(hor, ver, outline, levels=[0.5], colors=[INK], linewidths=1.0)
    ax.set_title(label + (r'   ($z = 1.5$ nm)' if plane == 'xy' else ''))
    ax.set_aspect('equal'); ax.grid(False)
    fig.colorbar(im, ax=ax, fraction=0.046)
fig.suptitle('Full strain tensor — the shear components were absent from the old model',
             fontsize=11, y=1.04)
fig.tight_layout(); plt.show()

d, dr = strain.at(pyr), strain.at_rms(pyr)
print(f"in the dot: trace {d['exx']+d['eyy']+d['ezz']:+.5f}, "
      f"biaxial {d['exx']+d['eyy']-2*d['ezz']:+.5f}")
print(f"rms shear: exy {dr['exy']:.5f}, eyz {dr['eyz']:.5f}, ezx {dr['ezx']:.5f}  "
      f"(means vanish by symmetry -- use at_rms, not at)")

## 3. The piezoelectric potential

Driven entirely by the shear components, so it was not computable at all before. $\varepsilon_{xy}$
changes sign under a 90° rotation while the pyramid does not, so $\phi$ is **odd** under that
rotation — the four-lobe pattern below. That is an exact constraint, verified to 8e-16.

This is what reduces the symmetry of the *scalar* problem from C4v to C2v.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13.5, 3.8))
zi_list = [iz_base + int(round(zz / H)) for zz in (0.5, 3.0)]
v = np.abs(phi).max() * 1e3
for ax, zi in zip(axes[:2], zi_list):
    im = ax.pcolormesh(cx, cx, phi[:, :, zi].T * 1e3, cmap=DIV, shading='nearest',
                       norm=TwoSlopeNorm(0.0, -v, v), rasterized=True)
    ax.contour(cx, cx, pyr[:, :, zi].T.astype(float), levels=[0.5],
               colors=[INK], linewidths=1.0)
    ax.plot([-12, 12], [-12, 12], color=MUTED, lw=1, ls=':')
    ax.plot([-12, 12], [12, -12], color=MUTED, lw=1, ls=':')
    ax.annotate('[110]', (9.5, 9.5), color=MUTED, fontsize=8)
    ax.annotate(r'[1$\bar{1}$0]', (9.5, -10.5), color=MUTED, fontsize=8)
    ax.set_xlabel('x (nm)'); ax.set_ylabel('y (nm)')
    ax.set_title(f'$\\phi$ in-plane, z = {cz[zi]:.1f} nm')
    ax.set_xlim(-13, 13); ax.set_ylim(-13, 13); ax.set_aspect('equal'); ax.grid(False)
    fig.colorbar(im, ax=ax, label='$\\phi$ (meV)', fraction=0.046)

ax = axes[2]
im = ax.pcolormesh(cx, cz, phi[:, iy0, :].T * 1e3, cmap=DIV, shading='nearest',
                   norm=TwoSlopeNorm(0.0, -v, v), rasterized=True)
ax.contour(cx, cz, pyr[:, iy0, :].T.astype(float), levels=[0.5], colors=[INK], linewidths=1.0)
ax.set_xlabel('x (nm)'); ax.set_ylabel('z (nm)'); ax.set_title('$\\phi$ vertical, y = 0')
ax.set_xlim(-14, 14); ax.set_ylim(-6, 12); ax.set_aspect('equal'); ax.grid(False)
fig.colorbar(im, ax=ax, label='$\\phi$ (meV)', fraction=0.046)
fig.tight_layout(); plt.show()

print(f"C4 antisymmetry error {pz.c4_antisymmetry_error(phi):.2e} (exact zero required)")
print(f"phi: {phi.min()*1e3:+.2f} .. {phi.max()*1e3:+.2f} meV globally, "
      f"{phi[pyr].min()*1e3:+.2f} .. {phi[pyr].max()*1e3:+.2f} meV in the dot")

## 4. Electron wavefunctions

One-band electron states with $m^* = 0.040$ (Pryor's strain-averaged mass), with and without
the piezoelectric potential. $|\psi|^2$ is a magnitude, so it uses the **sequential** one-hue
ramp.

Without piezoelectricity the pyramid is exactly C4v and the $p$ doublet is degenerate to
**0.000 meV** — the two $p$ lobes point along $x$ and $y$ and are interchangeable. Switching
the piezoelectric field on splits them by a few meV and rotates them onto the $\langle110\rangle$
diagonals.

In [ ]:
Ve, Vh, m_e, m_h = pr.band_edge_fields(pyr, tr_new, mass_case='strain_averaged')
states = {}
for label, V in (('no piezo', Ve), ('piezo', Ve - phi)):   # electron energy = -e*phi
    E, psi, _ = eig.solve_lowest(qd.build_hamiltonian(m_e, V, H), k=6,
                                 tol=1e-8, maxiter=8000)
    dens = [ (psi[:, i]**2).reshape(X.shape) for i in range(4) ]
    dens = [ d / (d.sum() * H**3) for d in dens ]
    states[label] = (E, dens)
    nb = int((E < GAAS_CB).sum())
    print(f"{label:>9s}: E0 = {E[0]:.5f} eV, binding {(GAAS_CB-E[0])*1e3:.1f} meV, "
          f"bound = {nb}, p-splitting {(E[2]-E[1])*1e3:.3f} meV")

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(11.5, 7.2))
zi = iz_base + int(round(1.0 / H))
names = ['$s$  (ground)', '$p$  (first)', '$p$  (second)']
for r, label in enumerate(['no piezo', 'piezo']):
    E, dens = states[label]
    for c in range(3):
        ax = axes[r, c]
        sl = dens[c][:, :, zi].T
        ax.pcolormesh(cx, cx, sl, cmap=SEQ, shading='nearest',
                      vmin=0, vmax=sl.max(), rasterized=True)
        ax.contour(cx, cx, pyr[:, :, zi].T.astype(float), levels=[0.5],
                   colors=[INK], linewidths=1.0)
        ax.plot([-11, 11], [-11, 11], color='white', lw=0.8, ls=':', alpha=0.7)
        ax.plot([-11, 11], [11, -11], color='white', lw=0.8, ls=':', alpha=0.7)
        ax.set_xlim(-11, 11); ax.set_ylim(-11, 11); ax.set_aspect('equal'); ax.grid(False)
        ax.set_title(f'{names[c]}   {E[c]*1e3:.1f} meV' if c else
                     f'{names[c]}   {E[c]:.4f} eV', fontsize=9)
        if c == 0:
            ax.set_ylabel(f'{label}\n\ny (nm)')
        if r == 1:
            ax.set_xlabel('x (nm)')
sp0 = (states['no piezo'][0][2] - states['no piezo'][0][1]) * 1e3
sp1 = (states['piezo'][0][2] - states['piezo'][0][1]) * 1e3
fig.suptitle(f'$|\\psi|^2$ at z = {cz[zi]:.1f} nm — p-doublet splitting '
             f'{sp0:.3f} meV without piezo, {sp1:.3f} meV with '
             f'(dotted lines are $\\langle110\\rangle$)', fontsize=10, y=1.0)
fig.tight_layout(); plt.show()

### Confinement along the growth axis

$|\psi|^2$ against the band edge, on the same axis. The ground state sits well inside the well
and is pushed toward the base by the taper — the well is deepest where the pyramid is widest.

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 4.2))
E, dens = states['piezo']
ax.plot(cz, Ve[ix0, iy0, :], color=MUTED, lw=1.6)
ax.annotate('$E_c$', (10.5, Ve[ix0, iy0, -1] + 0.01), color=MUTED, fontsize=9)
ax.axhline(GAAS_CB, color=GRIDC, lw=1)
for i, (c, nm) in enumerate(zip([S1, S2, S3], ['$s$', '$p_1$', '$p_2$'])):
    prof = dens[i][ix0, iy0, :]
    ax.plot(cz, E[i] + prof * 0.45 / dens[0].max(), color=c)
    ax.axhline(E[i], color=c, lw=0.8, ls='--', alpha=0.5)
    ax.annotate(nm, (-5.4, E[i] + 0.012), color=c, fontsize=9, fontweight='bold')
ax.axvspan(0, BASE / 2, color=GRIDC, alpha=0.5, zorder=0, lw=0)
ax.set_xlabel('z (nm)'); ax.set_ylabel('energy (eV)')
ax.set_title('Electron states and the conduction band edge, x = y = 0')
ax.set_xlim(-6, 12); ax.set_ylim(1.05, 1.62)
fig.tight_layout(); plt.show()

## 5. Multiband hole states

The four-band hole ground state, resolved by Bloch band. The hole is dominantly heavy-hole, as
expected under compressive biaxial strain — the Bir-Pikus $Q_\varepsilon$ term pushes HH above
LH by $2|Q_\varepsilon|$.

Note the hole $p$ doublet is split **even without piezoelectricity** (3.0 meV four-band,
9.4 meV six-band): the k·p valence structure and the Bir-Pikus shear terms already make
$[110]$ and $[1\bar10]$ inequivalent. That is the one place where the scalar one-band picture
above is misleading.

In [ ]:
Hm, PADm = 1.0, 8.0
cxm, czm, Xm, Ym, Zm, pyrm = clean_grid(Hm, PADm)
strain_m = sf.solve_strain(pyrm, eps_star, *CIJ, Hm)
phi_m = pz.potential(strain_m, np.where(pyrm, dot['e14'], matrix['e14']), EPS_R, Hm)
ops = kpc.GridOperators(Xm.shape, Hm, periodic=False)
Vem, Vhm, _, _ = pr.band_edge_fields(pyrm, strain_m.trace)

f4 = kp.material_fields(pyrm, dot, matrix, Vhm + phi_m, Vem - phi_m, n_bands=4)
H4 = kp.confined_hamiltonian(ops, f4, n_bands=4, hole_convention=True, strain=strain_m)
E4, V4, _ = eig.solve_lowest(H4, k=6, tol=1e-8, maxiter=8000)
n = Xm.size
psi4 = V4[:, 0].reshape(4, n)
weights = (np.abs(psi4)**2).sum(axis=1)
weights /= weights.sum()
dens4 = (np.abs(psi4)**2).sum(axis=0).reshape(Xm.shape)
dens4 /= dens4.sum() * Hm**3
print(f"4-band hole ground state {-E4[0]*1e3:.2f} meV above the GaAs VB edge")
print("band composition:", {l: f"{w:.3f}" for l, w in zip(kp.BAND_LABELS[4], weights)})

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.8))
zim = int(np.argmin(np.abs(czm - 1.0)))
ax = axes[0]
sl = dens4[:, :, zim].T
ax.pcolormesh(cxm, cxm, sl, cmap=SEQ, shading='nearest', vmin=0, vmax=sl.max(),
              rasterized=True)
ax.contour(cxm, cxm, pyrm[:, :, zim].T.astype(float), levels=[0.5], colors=[INK], lw=1.0)
ax.set_xlabel('x (nm)'); ax.set_ylabel('y (nm)'); ax.set_aspect('equal'); ax.grid(False)
ax.set_title(f'$|\\psi|^2$ in-plane, z = {czm[zim]:.1f} nm')
ax.set_xlim(-11, 11); ax.set_ylim(-11, 11)

ax = axes[1]
sl = dens4[:, len(cxm)//2, :].T
ax.pcolormesh(cxm, czm, sl, cmap=SEQ, shading='nearest', vmin=0, vmax=sl.max(),
              rasterized=True)
ax.contour(cxm, czm, pyrm[:, len(cxm)//2, :].T.astype(float), levels=[0.5],
           colors=[INK], lw=1.0)
ax.set_xlabel('x (nm)'); ax.set_ylabel('z (nm)'); ax.set_aspect('equal'); ax.grid(False)
ax.set_title('$|\\psi|^2$ vertical, y = 0')
ax.set_xlim(-14, 14); ax.set_ylim(-6, 12)

ax = axes[2]
bars = ax.bar(range(4), weights, color=[S1, S2, S1, S2], width=0.62)
for i, w in enumerate(weights):
    ax.annotate(f'{w:.3f}', (i, w), ha='center', va='bottom', fontsize=8, color=INK2)
ax.set_xticks(range(4)); ax.set_xticklabels(kp.BAND_LABELS[4], rotation=30, ha='right')
ax.set_ylabel('probability weight'); ax.set_ylim(0, max(weights) * 1.25)
ax.set_title('Bloch-band composition (HH dominates)')
ax.grid(axis='x', visible=False)
fig.tight_layout(); plt.show()

## What these figures show

- **Band alignment.** The corrected well tapers from base to apex and leaks strain into the
  barrier; the old model was a flat plateau with an identically unstrained matrix, and 17%
  too deep a shift.
- **Strain.** All six components exist now. The shear rms is ~13% of the trace.
- **Piezoelectricity.** A four-lobe, C4-odd potential of ±56 meV globally and ±22 meV inside
  the dot.
- **Electrons.** The $p$ doublet is degenerate to 0.000 meV on a clean grid without
  piezoelectricity, and splits by a few meV with it. On an `np.arange` grid at h = 0.7 the
  splitting is nonzero from grid asymmetry alone, which is why the guards in `clean_grid` are
  asserted rather than assumed.
- **Holes.** Dominantly heavy-hole under compressive strain, and already C2v-split before any
  piezoelectric field.

Open, and visible in these plots: the electron is **overbound** relative to Pryor (~150 meV
binding against ~110, and three bound states where he reports one). Correcting the geometry
made that worse, not better, and the piezoelectric field moves $E_0$ by only 0.2 meV. See
`strain_corrections.ipynb` for the full accounting.